# 06 UniversalRAG T5Gemma router: light vs visual@100 (Kaggle T4)

Trains the UniversalRAG T5Gemma-2-270M multi-label router (wgcyeo/UniversalRAG, pinned commit) on our labels, 5-fold CV on our fixed folds (seed 42).

- Two arms only (their router has no depth): light vs visual@100 (ColVec re-sort of light's top 100).
- Labels in their vocabulary: `visual_helps` (visual@100 R@10 > light R@10) -> `image`, else `paragraph` (ties count as light).
- Decision: their `route_batch` with their 0.8 threshold; `image` or `table` selected -> visual@100, else light.
- Their trainer is unchanged apart from fp32 instead of bf16 (T4 has no bf16) and wandb off (their non-wandb path logs to tensorboard).
- Batch: their effective batch 64 is kept as 4 per step x 16 accumulation (their CLI flags) because 16 per step runs out of memory on a T4 in fp32.

Input: dataset with `labels_ladder_k100.jsonl`. Settings: Accelerator GPU T4, Internet ON, secret `HF_TOKEN` (the base model is gated: accept its license on Hugging Face first).
Needs transformers >= 5 (their README), so run it in its own session, never after notebook 05.
Output: `/kaggle/working/routing_out_06.zip`.

## 1. Settings

In [ ]:
INPUT_NAME = "labels_ladder_k100.jsonl"
OUT = "/kaggle/working/routing_out"
ZIP = "routing_out_06.zip"
REPO = "https://github.com/wgcyeo/UniversalRAG"
COMMIT = "d29ab3acaebfbaebdc7f095457076585bec88065"
CHECKOUT = "/kaggle/working/UniversalRAG"

MODEL = "google/t5gemma-2-270m-270m"    # their default
LABEL_FOR = {False: "paragraph", True: "image"}   # keyed by visual_helps
THRESHOLD = 0.8                          # their default
TRAIN_ARGS = ["--batch-size", "4", "--gradient-accumulation-steps", "16"]   # their effective batch 64 (16 x 4) kept; 16 per step OOMs a T4 in fp32. Other defaults: 5 epochs, lr 2e-5, LoRA r 32

## 2. Install, clone, patch
Kaggle's preinstalled torch is kept (do not reinstall torch). Kaggle's torchao is removed: the repo does not use it and peft rejects the old version. transformers >= 5 is required by their README for T5Gemma 2; held below 5.15 because 5.15 removed `TrainingArguments(warmup_ratio=...)`, which their trainer passes.

In [ ]:
!pip uninstall -y -q torchao
!pip install -q "transformers>=5,<5.15" "peft>=0.18" "accelerate>=1.2" safetensors tabulate tensorboard wandb

In [ ]:
import glob
import json
import os
import subprocess
import sys

try:
    from kaggle_secrets import UserSecretsClient
    os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
    print("HF_TOKEN loaded from Kaggle secrets")
except Exception:
    print("no HF_TOKEN secret; gated models will fail to download")

found = sorted(glob.glob(f"/kaggle/input/**/{INPUT_NAME}", recursive=True))
if not found:
    raise SystemExit(f"attach a dataset containing {INPUT_NAME} (from data/work/ondemand_v2/<fp>/routing/)")
INPUT = found[0]
rows = [json.loads(line) for line in open(INPUT)]
print(INPUT, len(rows), "queries", sorted({r["fold"] for r in rows}), "folds")
if len(rows) not in (214, 220) or "visual100_recall@10" not in rows[0]:
    raise SystemExit("wrong input file: expected the ladder labels (214 queries, or 220 before the 6 flawed were dropped)")
os.makedirs(f"{OUT}/decisions", exist_ok=True)
os.makedirs(f"{OUT}/logs", exist_ok=True)

import torch
if not torch.cuda.is_available():
    raise SystemExit("no GPU: set Accelerator to GPU T4 in the notebook settings")
print(sys.version.split()[0], "torch", torch.__version__, torch.cuda.get_device_name(0), torch.cuda.get_device_capability(0))

In [ ]:
def run(cmd, log, cwd, env=None):
    with open(log, "w") as sink:
        code = subprocess.run(cmd, cwd=cwd, env={**os.environ, "CUDA_VISIBLE_DEVICES": "0", "PYTHONPATH": cwd, **(env or {})},
                              stdout=sink, stderr=subprocess.STDOUT).returncode
    if code:
        print(open(log).read()[-3000:])
        raise SystemExit(f"failed ({code}): {' '.join(cmd)}; full log in {log}")

if not os.path.exists(CHECKOUT):
    subprocess.run(["git", "clone", "-q", REPO, CHECKOUT], check=True)
subprocess.run(["git", "-C", CHECKOUT, "checkout", "-q", "--", "."], check=True)
subprocess.run(["git", "-C", CHECKOUT, "checkout", "-q", COMMIT], check=True)

patches = {
    "route/train/train_t5gemma.py": [
        ("torch_dtype=torch.bfloat16 if torch.cuda.is_available() else torch.float32,", "torch_dtype=torch.float32,"),
        ("bf16=torch.cuda.is_available(),", "bf16=False,"),
        ('parser.add_argument("--use-wandb", action="store_true", default=True,', 'parser.add_argument("--use-wandb", action="store_true", default=False,'),
    ],
    "route/train/route_t5gemma.py": [
        ('torch_dtype=torch.bfloat16 if self.device == "cuda" else torch.float32,', "torch_dtype=torch.float32,"),
    ],
}
for name, edits in patches.items():
    path = f"{CHECKOUT}/{name}"
    text = open(path).read()
    for old, new in edits:
        if old not in text:
            raise SystemExit(f"patch target not found in {name}: {old}")
        text = text.replace(old, new)
    open(path, "w").write(text)
print(subprocess.run(["git", "-C", CHECKOUT, "diff"], capture_output=True, text=True).stdout)

## 3. Preflight: their modules import and the gated base model is reachable

In [ ]:
check = """
import torch, transformers, peft
from transformers import AutoConfig, AutoTokenizer
import route.train.train_t5gemma
from route.train.route_t5gemma import T5GemmaMultiLabelRouter
from route.train.router_common import LABELS
from peft.import_utils import is_torchao_available
is_torchao_available()
print("transformers", transformers.__version__, "peft", peft.__version__, "labels", LABELS)
AutoConfig.from_pretrained(%r)
AutoTokenizer.from_pretrained(%r)
print("base model reachable")
""" % (MODEL, MODEL)
run(["python", "-c", check], f"{OUT}/logs/preflight.log", CHECKOUT)
print(open(f"{OUT}/logs/preflight.log").read()[-2000:])

## 4. Driver for routing a fold

In [ ]:
%%writefile /kaggle/working/t5_route.py
import json
import sys

from route.train.route_t5gemma import T5GemmaMultiLabelRouter

model_path, source, fold, threshold, out = sys.argv[1:6]
rows = [json.loads(line) for line in open(source) if json.loads(line)["fold"] == int(fold)]
router = T5GemmaMultiLabelRouter(model_path)
predictions = router.route_batch([r["query"] for r in rows], threshold=float(threshold), return_scores=True)
with open(out, "w") as sink:
    for r, (labels, scores) in zip(rows, predictions):
        branch = "visual100" if {"image", "table"} & set(labels) else "light"
        sink.write(json.dumps({"query_id": r["query_id"], "branch": branch, "fold": int(fold),
                               "labels": labels, "scores": scores}) + "\n")

## 5. 5-fold CV
Resumable per fold; each fold's checkpoint is deleted after routing.

In [ ]:
import shutil

tag = "universalrag-t5gemma"
parts = []
for fold in sorted({r["fold"] for r in rows}):
    part = f"{OUT}/folds/{tag}_fold{fold}.jsonl"
    os.makedirs(os.path.dirname(part), exist_ok=True)
    if not os.path.exists(part):
        data = f"/kaggle/working/t5_train_fold{fold}.json"
        json.dump([{"question": r["query"], "gt_retrieval": LABEL_FOR[r["visual_helps"]]} for r in rows if r["fold"] != fold],
                  open(data, "w"), indent=1)
        model_dir = f"/kaggle/working/t5_fold{fold}"
        run(["python", "route/train/train_t5gemma.py", "--model-name", MODEL, "--train-data", data, "--output-dir", model_dir, *TRAIN_ARGS],
            f"{OUT}/logs/{tag}_train_fold{fold}.log", CHECKOUT, {"PYTORCH_CUDA_ALLOC_CONF": "expandable_segments:True"})
        run(["python", "/kaggle/working/t5_route.py", model_dir, INPUT, str(fold), str(THRESHOLD), part],
            f"{OUT}/logs/{tag}_route_fold{fold}.log", CHECKOUT)
        shutil.rmtree(model_dir)
        print("done fold", fold)
    parts.append(part)
with open(f"{OUT}/decisions/{tag}.jsonl", "w") as sink:
    for part in parts:
        sink.write(open(part).read())

## 6. Summary and download

In [ ]:
by_qid = {r["query_id"]: r for r in rows}
alias = {"visual": "visual100"}
pages = {"light": 0, "visual20": 20, "visual50": 50, "visual100": 100}
for path in sorted(glob.glob(f"{OUT}/decisions/*.jsonl")):
    decisions = [json.loads(line) for line in open(path)]
    arms = [alias.get(d["branch"], d["branch"]) for d in decisions]
    recall = sum(by_qid[d["query_id"]][f"{a}_recall@10"] for d, a in zip(decisions, arms)) / len(decisions)
    counts = {a: arms.count(a) for a in pages}
    print(f"{os.path.basename(path):45s} n={len(decisions)} R@10={recall:.2f} pages={sum(pages[a] for a in arms) / len(arms):.1f} {counts}")
subprocess.run(["zip", "-qr", f"/kaggle/working/{ZIP}", "routing_out"], cwd="/kaggle/working", check=True)
print(f"download /kaggle/working/{ZIP} (Output panel on the right)")